# Assignment 1 — Pixel Geometry

Implements adjacency, connected components, and reusable pixel-neighbour helpers.

This notebook is intentionally modular. **Style 1** uses OpenCV operations; **Style 2** uses OpenCV only for reading and Matplotlib for display, while the image-processing logic is written with arrays, loops, queues, and formulas.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LAB_DIR = Path.cwd()

def show_images(images, titles, cmap="gray", figsize=(14, 4)):
    """Display a sequence of images consistently in a notebook."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    axes = np.atleast_1d(axes)
    for axis, image, title in zip(axes, images, titles):
        axis.imshow(image, cmap=cmap)
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()


## Style 1 — OpenCV built-in operations

In [2]:
def neighbours_cv(image, point, adjacency="8", values={0, 1}):
    """Return value-matching neighbours using padded image indexing."""
    x, y = point
    offsets = [(a, b) for a in (-1, 0, 1) for b in (-1, 0, 1) if (a, b) != (0, 0)]
    if adjacency == "4":
        offsets = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    result = []
    for dx, dy in offsets:
        nx, ny = x + dx, y + dy
        if 0 <= nx < image.shape[0] and 0 <= ny < image.shape[1] and image[nx, ny] in values:
            result.append(((nx, ny), int(image[nx, ny])))
    return result

def connected_components_cv(binary, connectivity=8):
    count, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary.astype(np.uint8), connectivity)
    return labels, stats[1:], centroids[1:]


## Style 2 — DSA/manual implementation

In [3]:
from collections import deque

def neighbours_manual(image, point, adjacency="8", values={0, 1}):
    x, y = point
    offsets = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if adjacency in ("8", "m"):
        offsets += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    output = []
    for dx, dy in offsets:
        nx, ny = x + dx, y + dy
        if 0 <= nx < image.shape[0] and 0 <= ny < image.shape[1] and image[nx, ny] in values:
            if adjacency != "m" or (dx == 0 or dy == 0 or not any(
                image[ax, ay] in values for ax, ay in
                [(x + dx, y), (x, y + dy)] if 0 <= ax < image.shape[0] and 0 <= ay < image.shape[1])):
                output.append(((nx, ny), int(image[nx, ny])))
    return output

def label_components_manual(binary, connectivity=8):
    labels = np.zeros(binary.shape, dtype=np.int32)
    offsets = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if connectivity == 8:
        offsets += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    label = 0
    for r in range(binary.shape[0]):
        for c in range(binary.shape[1]):
            if binary[r, c] and labels[r, c] == 0:
                label += 1
                queue = deque([(r, c)])
                labels[r, c] = label
                while queue:
                    x, y = queue.popleft()
                    for dx, dy in offsets:
                        nx, ny = x + dx, y + dy
                        if (0 <= nx < binary.shape[0] and 0 <= ny < binary.shape[1]
                                and binary[nx, ny] and labels[nx, ny] == 0):
                            labels[nx, ny] = label
                            queue.append((nx, ny))
    return labels, label


## Demonstration

In [4]:
image = np.array([[0, 1, 2, 0], [1, 1, 0, 2], [2, 0, 1, 1], [0, 2, 1, 0]], np.uint8)
print("8-neighbours (OpenCV-style):", neighbours_cv(image, (1, 1), "8"))
print("8-neighbours (manual):", neighbours_manual(image, (1, 1), "8"))
binary = (image == 1).astype(np.uint8)
print("Manual component count:", label_components_manual(binary, 8)[1])


8-neighbours (OpenCV-style): [((0, 0), 0), ((0, 1), 1), ((1, 0), 1), ((1, 2), 0), ((2, 1), 0), ((2, 2), 1)]
8-neighbours (manual): [((0, 1), 1), ((2, 1), 0), ((1, 0), 1), ((1, 2), 0), ((0, 0), 0), ((2, 2), 1)]
Manual component count: 1
